# Lesson 01 - Introducción a los Agentes de IA

¡Bienvenido a la primera lección del curso **Agentes de IA para Principiantes**!

Un **agente de IA** es un programa que utiliza un modelo de lenguaje grande (LLM) como su motor de razonamiento y puede tomar *acciones* en el mundo real — llamar a APIs, consultar bases de datos o ejecutar código — para lograr un objetivo en nombre de un usuario.

En este cuaderno construirás tu primer agente: un **Agente de Viajes** que recomienda destinos de vacaciones. A lo largo del camino aprenderás a:

1. Conectarte al Servicio Azure AI Foundry Agent usando el **Marco de Agentes de Microsoft**.
2. Darle al agente una **herramienta** — una función sencilla en Python que puede llamar.
3. Ejecutar el agente e inspeccionar su respuesta.
4. Transmitir la respuesta del agente token por token.


## Configuración

Antes de ejecutar este cuaderno, asegúrate de tener:

1. **Un proyecto de Azure AI Foundry** con un modelo de chat desplegado (por ejemplo, `gpt-4o-mini`).
2. **Has iniciado sesión con el Azure CLI** — ejecuta `az login` en tu terminal.
3. **Configuradas las variables de entorno requeridas:**
   - `AZURE_AI_PROJECT_ENDPOINT` — el endpoint de tu proyecto Azure AI Foundry.
   - `AZURE_AI_MODEL_DEPLOYMENT_NAME` — el nombre de tu modelo desplegado.

La celda a continuación instala los paquetes de Python que necesitas.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Creando tu Primer Agente

Un agente necesita dos cosas:

- **Instrucciones** que le digan *quién es* y *cómo comportarse* (un mensaje del sistema).
- **Herramientas** — funciones de Python decoradas con `@tool` que el agente puede llamar para obtener información o realizar acciones.

A continuación definimos una herramienta sencilla que devuelve una lista de destinos turísticos populares. El agente usará esta herramienta cuando un usuario pida recomendaciones de viaje.


In [ ]:
@tool(approval_mode="never_require")
def get_destinations() -> list[str]:
    """Get a list of popular vacation destinations."""
    return [
        "Barcelona",
        "Paris",
        "Berlin",
        "Tokyo",
        "Sydney",
        "New York City",
        "Cairo",
        "Cape Town",
        "Rio de Janeiro",
        "Bali",
    ]

In [ ]:
agent = await provider.create_agent(
    tools=[get_destinations],
    name="TravelAgent",
    instructions=(
        "You are a helpful travel agent. Help users find their perfect vacation "
        "destination based on their preferences. Use the get_destinations tool "
        "to see available destinations."
    ),
)

response = await agent.run(
    "I'm looking for a warm beach destination. What do you recommend?"
)
print(response)

## Respuestas en streaming

Para una experiencia más interactiva, puedes **transmitir** la respuesta del agente. En lugar de esperar la respuesta completa, el agente entrega fragmentos de texto a medida que se generan. Esto es especialmente útil en interfaces de chat donde deseas mostrar la salida en tiempo real.


In [ ]:
async for chunk in agent.run(
    "Tell me about Tokyo as a travel destination", stream=True
):
    print(chunk, end="", flush=True)

## Resumen

En esta lección aprendiste a:

- **Crear un proveedor** que se conecta al servicio Azure AI Foundry Agent mediante `AzureAIProjectAgentProvider`.
- **Definir una herramienta** usando el decorador `@tool` para que el agente pueda llamar a tus funciones en Python.
- **Ejecutar el agente** con un mensaje del usuario e imprimir su respuesta.
- **Transmitir respuestas** para salida en tiempo real.

En la próxima lección exploraremos los marcos agenticos con mayor profundidad y aprenderemos cómo proporcionar a los agentes herramientas más potentes y capacidades de razonamiento en múltiples pasos.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Aviso Legal**:  
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda la traducción profesional realizada por un humano. No nos responsabilizamos por malentendidos o interpretaciones erróneas derivadas del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
